i now want to visualise the data we have gathered.  
We start simple, stack all neurons, see which one is acting where

We want to see the common neurons, and the uncommon ones, etc. between dog/fox and cat.   

Basically we want to know what makes 4e:55 decide something is snout vs cat.  

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from PIL import Image
from lucent.optvis import render, param, transform, objectives
import shelve
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import random

import matplotlib.pyplot as plt
import numpy as np
from lucent.modelzoo import inceptionv1
from PIL import Image
from torch.nn import functional as F
from lucent.optvis import param

from olt.act import InputOutputModelSnapshot
import json

from lucent.modelzoo import inceptionv1
from olt.tfms import transform
from olt.act import InputOutputModelSnapshot
from olt.show import show_single_channel_red_green_black as S


base_report_dir = Path("mass-train-reports")
plt.style.use("default")


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


FLAT_IMAGE_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/flat-images"
)

In [ ]:

with open('fox-neurons-description.json') as f:
    fox_neurons_db = json.load(f)

with open('cat-neurons-description.json') as f:
    cat_neurons_db = json.load(f)

In [ ]:
# 528 in channels
# we want to stack them lol, idk

import matplotlib.pyplot as plt
import numpy as np

def render_grid(data, shape=(22, 24), cell_size=0.6, fontsize=8, text_color="black"):
    """
    data: list of dicts like {"color": "yellow"/"green"/"red"/"black", "text": "..."}
    shape: (rows, cols) to reshape the flat list into
    """
    rows, cols = shape
    assert len(data) == rows * cols, f"Expected {rows*cols} items for shape {shape}, got {len(data)}"

    colors = np.array([d["color"] for d in data]).reshape(rows, cols)
    texts = np.array([d["text"] for d in data]).reshape(rows, cols)

    fig, ax = plt.subplots(figsize=(cols * cell_size, rows * cell_size))

    for r in range(rows):
        for c in range(cols):
            ax.add_patch(plt.Rectangle((c, rows - r - 1), 1, 1,
                                        facecolor=colors[r, c], edgecolor="gray", linewidth=0.5))
            ax.text(c + 0.5, rows - r - 1 + 0.5, texts[r, c],
                    ha="center", va="center", fontsize=fontsize, color=text_color)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
import ast

# mixed4d: 447
# 1x1: 112 (112)
# 3x3: 288 (400)
# 5x5: 64 (464)
# pool: 64 (528)

def _get_flattened_chan_in_cat_layer(layer_name, channel_in_layer):
    if layer_name == "mixed4d_1x1_pre_relu_conv":
        return channel_in_layer
    elif layer_name == "mixed4d_3x3_pre_relu_conv":
        return 112 + channel_in_layer
    elif layer_name == "mixed4d_5x5_pre_relu_conv":
        return 400 + channel_in_layer
    elif layer_name == "mixed4d_pool_reduce_pre_relu_conv":
        return 464 + channel_in_layer
    else:
        raise Exception(f"invalid layer name: {layer_name}")

def get_chan_by_comps(neurons_db, prefix):
    chan_by_text = {}
    for v in neurons_db.values():
        layer_name, channel, cluster_id = v["key"]
        flat_chan = _get_flattened_chan_in_cat_layer(layer_name, channel)
        meaning = v["meaning"]
        chan_by_text.setdefault(flat_chan, []).append((prefix, cluster_id, meaning))
    return chan_by_text


In [ ]:
import textwrap

fox_chan_by_comps = get_chan_by_comps(fox_neurons_db, "fox")
cat_chan_by_comps = get_chan_by_comps(cat_neurons_db, "cat")

def _get_comps_and_text_from(chan, chan_by_comps):
    if chan not in chan_by_comps:
        return [], False
    return chan_by_comps[chan], True

def _tw(word):
    return "\n".join(textwrap.wrap(word, 13))

def _to_text(pref, entries):
    # entries: list of (cluster_id, meaning)
    bullets = "\n".join(f"- {_tw(meaning)} ({cluster_id})" for cluster_id, meaning in entries)
    return f"{_tw(pref)}\n{bullets}"

chan_by_text_and_color = {}
for chan in range(528):
    cat_entries, cat_found = _get_comps_and_text_from(chan, cat_chan_by_comps)
    fox_entries, fox_found = _get_comps_and_text_from(chan, fox_chan_by_comps)

    cat_meanings = [(cid, meaning) for _, cid, meaning in cat_entries]
    fox_meanings = [(cid, meaning) for _, cid, meaning in fox_entries]

    cat_cids = {cid for cid, _ in cat_meanings}
    fox_cids = {cid for cid, _ in fox_meanings}

    if fox_found and cat_found:
        if cat_cids & fox_cids:
            if cat_cids & fox_cids:
                chan_by_text_and_color[chan] = {
                    "color": "red",
                    "text": _to_text("cat", cat_meanings) + "\n---\n" + _to_text("fox", fox_meanings),
                }
        else:
            chan_by_text_and_color[chan] = {
                "color": "yellow",
                "text": _to_text("cat", cat_meanings) + "\n---\n" + _to_text("fox", fox_meanings),
            }
    elif fox_found and not cat_found:
        chan_by_text_and_color[chan] = {
            "color": "cyan",
            "text": _to_text("fox", fox_meanings),
        }
    elif not fox_found and cat_found:
        chan_by_text_and_color[chan] = {
            "color": "green",
            "text": _to_text("cat", cat_meanings),
        }
    else:
        chan_by_text_and_color[chan] = {
            "color": "black",
            "text": "",
        }

In [ ]:
for k in cat_neurons_db:
    # if "206" in k:
    #     print(k)
    if "'mixed4d_pool_reduce_pre_relu_conv', 29" in k:
        print(k)
print("--------")
for k in fox_neurons_db:
    # if "206" in k:
    #     print(k)
    if "'mixed4d_pool_reduce_pre_relu_conv', 29" in k:
        print(k)

In [ ]:
# 493 // 11 -> row = 44
# col = 9

In [ ]:
text_and_color = [chan_by_text_and_color[c] for c in range(528)]

render_grid(text_and_color, shape=(22, 24), cell_size=1.5)
plt.show()

In [ ]:
df = pd.read_csv("main_df.csv")

In [ ]:
fox_df = df[(df.cluster_label == 60) & (df.channel == 55) & (df.layer_name == "mixed4e_1x1_pre_relu_conv")].reset_index()
cat_df = df[(df.cluster_label == 61) & (df.channel == 55) & (df.layer_name == "mixed4e_1x1_pre_relu_conv")].reset_index()

len(fox_df), len(cat_df)

In [ ]:
model.get_submodule("mixed5b_5x5_pre_relu_conv").weight.shape

In [ ]:
def _get_ip_acts_for_neuron(ikey, layer_name, y, x):
    timg = transform(Image.open(FLAT_IMAGE_DIR / f"{ikey}.jpeg"))[None]
    return InputOutputModelSnapshot.get_activations(timg, model, [layer_name])[layer_name]["input"][0, :, y, x]

def _get_ip_acts_of_row(row):
    return _get_ip_acts_for_neuron(row.input_image_key, row.layer_name, row.y_position, row.x_position)

In [ ]:
from sklearn.preprocessing import normalize

def _norm(val):
    orig_shape_single_dim = False
    if len(val.shape) == 1:
        orig_shape_single_dim = True
        val = val.reshape(1, -1)
    res = normalize(val, "l2")
    if orig_shape_single_dim:
        res = res.reshape(-1)
    return res

In [ ]:
model = model.to("cpu")

In [ ]:
w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().cpu().reshape(-1)
fox_row = fox_df.iloc[0]
cat_row = cat_df.iloc[0]
fox_act = _get_ip_acts_of_row(fox_row)
cat_act = _get_ip_acts_of_row(cat_row)

In [ ]:
fp, cp = [_norm(a * w).reshape(22,24) for a in [fox_act, cat_act]]
# S(, 10)
S([fp, cp], 10)
plt.show()

In [ ]:
fox_act.shape,  fp.shape

In [ ]:
gfox_act = fox_act.reshape(22,24)
gcat_act = cat_act.reshape(22,24)

In [ ]:
S([gfox_act, gcat_act])

In [ ]:
fp[20, 13] = 0.
cp[20, 13] = 0.
gfox_act[20, 13] = 0.
gcat_act[20, 13] = 0.

In [ ]:
# once we remove the top, it is quite different.  
S([gfox_act, gcat_act], 10)
plt.show()

In [ ]:
# once we remove the top, it is quite different.  
S([fp, cp], 10)
plt.show()

(19, 21) bright spot in  bottom is animal-face cluster, common in both (both has the same cluster also)

In [ ]:
#19,21
# 19*24 + 21 = 477

In [ ]:
??_get_flattened_chan_in_cat_layer

In [ ]:
# high negative in fox
gfox_act[8,3]
# 8*24+3=195

# mixed4d_3x3_pre_relu_conv:83

In [ ]:
model = model.to("mps")
noise_acts = []

for batch in tqdm(list(batched(test_images, 8))):
    batch = torch.stack(batch).to("mps")
    act = InputOutputModelSnapshot.get_activations(batch, model, ["mixed4d_3x3_pre_relu_conv"])["mixed4d_3x3_pre_relu_conv"]["output"]
    act = act[:, 83, :, :].reshape(act.shape[0], -1)
    idx = torch.randint(0, act.shape[1], (act.shape[0],))
    samples = act[torch.arange(8), idx]  # shape [8]
    noise_acts.append(samples)
    # print(samples.shape)
    # break
noise_acts = torch.cat(noise_acts)

In [ ]:
from lucent.optvis import render

model = model.to("cpu")
svizs = render.render_vis(model, "mixed4d_3x3_pre_relu_conv:83")

In [ ]:
plt.imshow(svizs[-1][0])
plt.show()

In [ ]:
# noise_acts = torch.cat(noise_acts)
xs = range(len(noise_acts))
plt.scatter(xs, noise_acts)

# plt.savefig("noise-29.jpeg")
plt.show()

In [ ]:
# this would just relu out
label_by_points = main_act_defdict['mixed4d_3x3_pre_relu_conv']['83']
_ = plot_grid(label_by_points, 4, 4)

In [ ]:
gfox_act[19, 21], gcat_act[19, 21]

In [ ]:
%reset out

In [ ]:

# this would just relu out
label_by_points = main_act_defdict['mixed4d_pool_reduce_pre_relu_conv']['13']
_ = plot_grid(label_by_points, 4, 4)

In [ ]:
fp[19,21] = 0
cp[19,21] = 0
S([fp, cp], 10)
plt.show()

In [ ]:
20*24 + 13

# 22, 24
# 20, 13

(20,13) is bright in BOTH.   

so that is 20*24 + 13 = 493

# Activation ranges

- for each input image key in df
  - get activations for all layers
  - for each layer + channel + position combo
      - get the output activation value
      - get the pointwise mult in next 4e:55

In [ ]:
layer_names_4d = [
    "mixed4d_1x1_pre_relu_conv",
    "mixed4d_3x3_pre_relu_conv",
    "mixed4d_pool_reduce_pre_relu_conv",
    "mixed4d_5x5_pre_relu_conv",
]
our_neuron_layer_name = "mixed4e_1x1_pre_relu_conv"

def collect_acts_for_one_image(image_key, df, model, our_neuron_layer_name, our_neuron_channel, our_neuron_weight, device):
    layer_by_chan_by_cluster_id_by_act = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    layer_by_chan_by_cluster_id_by_pw = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    batch = transform(Image.open(FLAT_IMAGE_DIR / f"{image_key}.jpeg"))[None].to(device)
    # df = df[df.input_image_key == image_key]
    if df.index.name != "input_image_key":
        df = df.set_index("input_image_key", drop=False)
    df = df.loc[[image_key]]

    acts = InputOutputModelSnapshot.get_activations(batch, model, layer_names_4d + [our_neuron_layer_name])
    # our_neuron_weight = model.get_submodule(our_neuron_layer_name).weight[our_neuron_channel].detach().cpu().reshape(-1)

    
    for tup in df.itertuples():
        if tup.layer_name == our_neuron_layer_name:
            continue
        layer_by_chan_by_cluster_id_by_act[tup.layer_name][str(tup.channel)][str(tup.cluster_label)].append(
            acts[tup.layer_name]["output"][0, tup.channel, tup.y_position, tup.x_position].item()
        )

        patch = acts[our_neuron_layer_name]["input"][0, :, tup.y_position, tup.x_position]
        pw = (our_neuron_weight * patch).reshape(-1)
        this_tups_pw = pw[_get_flattened_chan_in_cat_layer(tup.layer_name, tup.channel)].item()
        layer_by_chan_by_cluster_id_by_pw[tup.layer_name][str(tup.channel)][str(tup.cluster_label)].append(
            this_tups_pw
        )

        
    return layer_by_chan_by_cluster_id_by_act, layer_by_chan_by_cluster_id_by_pw

In [ ]:
def collect_for_one_neuron(image_key, df, model, our_neuron_layer_name, our_neuron_channel, our_neuron_weight, device):
    layer_by_chan_by_cluster_id_by_act = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    if df.index.name != "input_image_key":
        df = df.set_index("input_image_key", drop=False)
    df = df.loc[[image_key]]
    df = df[(df.layer_name == our_neuron_layer_name) & (df.channel == our_neuron_channel) & (df.cluster_label != -1)]

    batch = transform(Image.open(FLAT_IMAGE_DIR / f"{image_key}.jpeg"))[None].to(device)
    acts = InputOutputModelSnapshot.get_activations(batch, model, [our_neuron_layer_name])
    

    
    for tup in df.itertuples():
        layer_by_chan_by_cluster_id_by_act[tup.layer_name][str(tup.channel)][str(tup.cluster_label)].append(
            acts[tup.layer_name]["output"][0, tup.channel, tup.y_position, tup.x_position].item()
        )

    return layer_by_chan_by_cluster_id_by_act

In [ ]:
df = pd.read_csv("main_df.csv")

In [ ]:
idf = df.set_index("input_image_key", drop=False)

In [ ]:
def merge_nested_dict(main, incoming):
    for layer_name, chan_dict in incoming.items():
        for chan, cluster_dict in chan_dict.items():
            for cluster_id, values in cluster_dict.items():
                main[layer_name][chan][cluster_id].extend(values)
    return main

## Build activations

In [ ]:
# our_neuron_layer_name = 
all_image_keys = df[
    (df.layer_name == our_neuron_layer_name) & 
    (df.channel == 55) & 
    (df.cluster_label != -1)
].input_image_key.unique()

In [ ]:
len(all_image_keys)

In [ ]:
from olt.shards import raw_iter_shards, read_image_shard

inet_label_dirs = [p for p in Path("image-shards").glob("*") if p.is_dir()]

test_images = []

for d in tqdm(inet_label_dirs):
    shards = raw_iter_shards(d)
    keys, images = next(read_image_shard(shards[0], 1, {}))
    key, image = keys[0], images[0]
    test_images.append(transform(image))

In [ ]:
from itertools import batched

In [ ]:
model = model.to("mps")
noise_acts = []

for batch in tqdm(list(batched(test_images, 8))):
    batch = torch.stack(batch).to("mps")
    act = InputOutputModelSnapshot.get_activations(batch, model, ["mixed4d_pool_reduce_pre_relu_conv"])["mixed4d_pool_reduce_pre_relu_conv"]["output"]
    act = act[:, 29, :, :].reshape(act.shape[0], -1)
    idx = torch.randint(0, act.shape[1], (act.shape[0],))
    samples = act[torch.arange(8), idx]  # shape [8]
    noise_acts.append(samples)
    # print(samples.shape)
    # break
noise_acts = torch.cat(noise_acts)

In [ ]:
# noise_acts = torch.cat(noise_acts)
xs = range(len(noise_acts))
plt.scatter(xs, noise_acts)

# plt.savefig("noise-29.jpeg")
plt.show()

In [ ]:
inet_labels = df.imagenet_label.unique()
rows = []


In [ ]:
len(rows)

In [ ]:
import json
import pickle
from itertools import batched
# start from 7, it had errored out there
model = model.to("cpu")
batches = list(batched(all_image_keys, 10))

# batches = batches
our_neuron_weight = model.get_submodule(our_neuron_layer_name).weight[55].detach().cpu().reshape(-1)

main_act_dict_55 = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
# main_pw_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

with torch.no_grad():
    for b, batch in enumerate(tqdm(batches)):
        for image_key in batch:
            act_dict = collect_for_one_neuron(image_key, idf, model, our_neuron_layer_name, 55, our_neuron_weight, "cpu")
            merge_nested_dict(main_act_dict_55, act_dict)

        plain_act_dict = json.loads(json.dumps(main_act_dict_55))
        
        with open("main_act_dict_55.pkl", "wb") as f:
            pickle.dump(plain_act_dict, f)

In [ ]:
import json
import pickle
# start from 7, it had errored out there
model = model.to("cpu")
batches = list(batched(all_image_keys, 1000))

batches = batches[7:]
our_neuron_weight = model.get_submodule(our_neuron_layer_name).weight[55].detach().cpu().reshape(-1)

main_act_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
main_pw_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

with torch.no_grad():
    for b, batch in enumerate(tqdm(batches)):
        for image_key in batch:
            act_dict, pw_dict = collect_acts_for_one_image(image_key, idf, model, our_neuron_layer_name, 55, our_neuron_weight, "cpu")
            merge_nested_dict(main_act_dict, act_dict)
            merge_nested_dict(main_pw_dict, pw_dict)

        plain_act_dict = json.loads(json.dumps(main_act_dict))
        plain_pw_dict = json.loads(json.dumps(main_pw_dict))

        with open("main_act_dict.pkl", "wb") as f:
            pickle.dump(plain_act_dict, f)
        with open("main_pw_dict.pkl", "wb") as f:
            pickle.dump(plain_pw_dict, f)

## Visualise

In [ ]:
import pickle

In [ ]:
import pickle
with open("main_pw_dict.pkl", "rb") as f:
    main_pw_dict = pickle.load(f)

with open("main_act_dict.pkl", "rb") as f:
    main_act_dict = pickle.load(f)

with open("main_pw_ckpt.pkl", "rb") as f:
    incoming_pw = pickle.load(f)

with open("main_act_ckpt.pkl", "rb") as f:
    incoming_act = pickle.load(f)


main_act_defdict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for k1, v1 in main_act_dict.items():
    for k2, v2 in v1.items():
        for k3, v3 in v2.items():
            main_act_defdict[k1][k2][k3] = v3

main_pw_defdict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for k1, v1 in main_pw_dict.items():
    for k2, v2 in v1.items():
        for k3, v3 in v2.items():
            main_pw_defdict[k1][k2][k3] = v3

In [ ]:
main_act_defdict = merge_nested_dict(main_act_defdict, incoming_act)
main_pw_defdict = merge_nested_dict(main_pw_defdict, incoming_pw)

In [ ]:
kelly_colors = [
    "#F2F3F4",  # white (skip as bg reference)
    "#F3C300", "#875692", "#F38400", "#A1CAF1", "#BE0032",
    "#C2B280", "#848482", "#008856", "#E68FAC", "#0067A5", "#F99379",
    "#604E97", "#F6A600", "#B3446C", "#DCD300", "#882D17", "#8DB600",
    "#654522", "#E25822", "#2B3D26",
]

def plot_grid(label_by_points, n_rows, n_cols, colors=None, figsize=None, sharey=True, max_points_per_label=200):
    label_by_points = {label: np.random.choice(points, min(len(points), max_points_per_label)) for label, points in label_by_points.items()}
    labels = list(label_by_points.keys())
    if colors is None:
        colors = kelly_colors[1:]
    color_map = {l: colors[i % len(colors)] for i, l in enumerate(labels)}

    if figsize is None:
        figsize = (7 * n_cols, 5 * n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, sharey=sharey)
    axes = np.array(axes).reshape(-1)  # flatten regardless of shape

    n_labels = len(labels)
    n_panels = n_rows * n_cols
    labels_per_panel = -(-n_labels // n_panels)  # ceil division

    for i, ax in enumerate(axes):
        panel_labels = labels[i * labels_per_panel : (i + 1) * labels_per_panel]
        for l in panel_labels:
            ys = label_by_points[l]
            ax.scatter(range(len(ys)), ys, color=color_map[l], label=l)
        if panel_labels:
            ax.legend()
    for ax in axes[n_labels:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig, axes

In [ ]:
plt.style.use("dark_background")

In [ ]:
# this would just relu out
label_by_points = main_act_defdict['mixed4d_pool_reduce_pre_relu_conv']['29']
_ = plot_grid(label_by_points, 4, 4)

In [ ]:
plt.style.use("dark_background")

In [ ]:
# cid_by_name = {
#     41: "Human Finger/Skin/Hand",
#     58: "Background",
#     53: "Dog Legs",
#     51: "Background-2",
#     24: "Letters",
#     59: "Car",
#     45: "Dog Stomach",
#     31: "Human Face",
#     43: "Food",
#     50: "Mountain",
#     48: "Human Leg Clothes",
#     60: "Fox",
#     61: "Cat",
#     40: "Mushroom",
#     57: "Green Grass",
#     32: "Human Faces (again)",
#     14: "Corn-like",
#     29: "Snake",
#     44: "Soup-like",
#     33: "Fish",
#     23: "Thin rod-like",
#     18: "Salamander",
#     55: "White background",
#     52: "Water",
#     46: "White dog stomach",
#     13: "Keyboard",
#     19: "Clock",
#     27: "Thin rod",
#     25: "Yellow food",
#     21: "Snake 2",
#     30: "Orange-like",
#     56: "dark-green background",
#     47: "green background 2",
#     12: "Knit clothes",
#     9: "Knit clothes 2",
#     36: "Repeated vertical lines",
#     5: "criss-crossing white rods",
#     26: "arc-like edge",
#     22: "curvy shape on cylinder",
#     16: "repeated vertical lines",
#     54: "arc on bowl",
#     35: "brown fur",
#     49: "pattern on cloth",
#     2: "Ruler-like",
#     6: "Cauliflower",
#     37: "Fence-like",
#     10: "Knit-3",
#     16: "Repeating grids",
#     28: "Rod 2",
#     7: "Yellow Flower",
#     3: "Window Grill",
#     39: "Sink-Like",
#     38: "Matchsticks like",
#     20: "Violet flower",
#     4: "spiral pattern",
#     17: "Repeating dots",
#     1: "Honey comb",
#     42: "Noodles",
    
# }

In [ ]:
model.get_submodule("mixed4d_pool_reduce_pre_relu_conv").bias[29]

In [ ]:
S([model.get_submodule("mixed4d_pool_reduce_pre_relu_conv").weight[29].detach().cpu().reshape(16,32)])

In [ ]:
cid_by_name = {
    146: "dog face",
    141: "human face",
    135: "car frame",
    138: "animal leg",
    144: "cat-like face",
    91: "background",
    128: "keyboard",
    124: "food",
    142: "small bird-face",
    56: "scaly skin",
    123: "dog body",
    116: "thin lines-like",
    120: "cup",
    129: "sky",
    99: "rod-like handle",
    100: "background 2",
    134: "clock",
    78: "letter",
    74: "bright yellow lights",
    140: "human face 2",
    132: "sky 2",
    137: "scaly skin 2",
    75: "sand like ground",
    81: "repeating dots",
    77: "small dense letters",
    109: "insect and green background",
    139: "bird face 2",
    113: "roof of some type",
    108: "orange",
    101: "sand like ground 2",
    93: "building arc",
    105: "messy grass",
    106: "insect and green background 2",
    110: "minar like",
    130: "water body",
    69: "letter 2",
    80: "letter 3",
    97: "handwash pump",
    126: "white surface?",
    53: "object in blue background",
    117: "human fingers",
    127: "frame of devices",
    42: "big lemon like",
    103: "arc",
    67: "cherry like",
    107: "big leaves",
    86: "fence like",
    133: "clock 2",
    51: "salad like?",
    33: "flower pattern on clothes",
    114: "camera like frame?",
    119: "tip on top of bigger object",
    145: "ferret face",
    143: "monkey face",
    79: "sphere",
    49: "animal part of some shape?",
    61: "matt-utensil like?",
    98: "caps?",
    76: "letter 4",
    65: "pen like",
    54: "dense grass 2",
    131: "mountain behind sea",
    21: "leapord print",
    30: "small drawings?",
    88: "hand rail like",
    50: "flower",
    112: "sink like dots",
    72: "honeycomb like",
    38: "circle of some size",
    25: "jail like",
    4: "turtle shell",
    46: "snake like",
    86: "pink flower",
    18: "knit cloth",
    45: "scaly skin 3",
    28: "corn like",
}

In [ ]:
name_by_med = {n: np.median(a) for n, a in name_by_acts.items()}

In [ ]:
only_pos_name = [n for (n, med) in name_by_med.items() if med > 0.]

In [ ]:
only_pos_name

In [ ]:
! pwd

In [ ]:
# label_by_points = main_act_dict_55[our_neuron_layer_name]['55']
fig, _ = plot_grid({n: a for (n,a) in name_by_acts.items() if n in only_pos_name}, 2, 2)
fig.savefig("only-positive.jpeg")
plt.close()

In [ ]:
name_by_acts = {
    cid_by_name[int(cid)]: acts
    for cid, acts in main_act_defdict['mixed4d_pool_reduce_pre_relu_conv']['29'].items() if int(cid) in cid_by_name
}
name_by_acts

In [ ]:
# label_by_points = main_act_dict_55[our_neuron_layer_name]['55']
fig, _ = plot_grid(name_by_acts, 4, 4)
fig.savefig("act-29.jpeg")
plt.close()

In [ ]:
fig.savefig("./small-activation-graphs-mixed4e-pre-relu-conv-55.jpeg")

In [ ]:
small_name_by_acts = {n: a for n, a in name_by_acts.items() if n not in ["Cat", "Car", "Fox"]}
fig, _ = plot_grid(small_name_by_acts, 4, 2)



In [ ]:
label_by_points = main_act_defdict['mixed4d_pool_reduce_pre_relu_conv']['29']
_ = plot_grid(label_by_points, 4, 4)

In [ ]:
_, ax = plt.subplots(1, 1)

for layer in main_act_defdict:
    for chan in main_act_defdict[layer]:
        ax.clear()
        for cluster_id in main_act_defdict[layer][chan]:
            acts = main_act_defdict[layer][chan]
            

In [ ]:
main_act_defdict['mixed4d_1x1_pre_relu_conv']['0'].keys()

# Check sparse signal neurons


493 has a very sparse signal, it is killing a whole lot of clusters. we would like to see if there are other neurons like this.   

In [ ]:
main_act_defdict["mixed4d_1x1_pre_relu_conv"]["92"]["16"]

In [ ]:
def get_sparse_ratio_for_neuron(label_by_acts):
    pos_meds, neg_meds = 0, 0
    for label, acts in label_by_acts.items():
        act_med = np.percentile(acts, 50)
        if act_med <= 0:
            neg_meds += 1
        else:
            pos_meds += 1
    return neg_meds / (pos_meds + neg_meds)


def get_max_cluster_median_for_neuron(label_by_acts):
    return np.max([np.percentile(acts, 50) for label, acts in label_by_acts.items()])

In [ ]:
get_max_cluster_median_for_neuron(main_act_defdict['mixed4d_pool_reduce_pre_relu_conv']['29'])

In [ ]:
get_sparse_ratio_for_neuron(main_act_defdict['mixed4d_pool_reduce_pre_relu_conv']['29'])

In [ ]:
# now we would like to check the max weight of a neuron, and compare with the ratio
# first flatten and get ratios

chan_by_ratio = {}
for layer_name in main_act_defdict:
    for channel in tqdm(main_act_defdict[layer_name], desc=layer_name):
        fchan = _get_flattened_chan_in_cat_layer(layer_name, int(channel))
        chan_by_ratio[fchan] = get_sparse_ratio_for_neuron(main_act_defdict[layer_name][channel])

In [ ]:
plt.hist(chan_by_ratio.values(), label="number clusters with negative medians / total clusters")
# plt.legend()
plt.show()

In [ ]:
chan_by_ratio

In [ ]:
# now we would like to check the max weight of a neuron, and compare with the ratio
# first flatten and get ratios

chan_by_max_med = {}
for layer_name in main_act_defdict:
    for channel in tqdm(main_act_defdict[layer_name], desc=layer_name):
        fchan = _get_flattened_chan_in_cat_layer(layer_name, int(channel))
        chan_by_max_med[fchan] = get_max_cluster_median_for_neuron(main_act_defdict[layer_name][channel])

In [ ]:
# now we would like to check the max weight of a neuron, and compare with the ratio
# first flatten and get ratios

layer_by_chan_by_ratio = defaultdict(lambda: defaultdict(dict))
for layer_name in main_act_defdict:
    for channel in tqdm(main_act_defdict[layer_name], desc=layer_name):
        fchan = _get_flattened_chan_in_cat_layer(layer_name, int(channel))
        # chan_by_ratio[fchan] = get_sparse_ratio_for_neuron(main_act_defdict[layer_name][channel])
        layer_by_chan_by_ratio[layer_name][channel] = get_sparse_ratio_for_neuron(main_act_defdict[layer_name][channel])

In [ ]:
layer_by_chan_by_ratio

In [ ]:
very_sparse_neurons = []
for layer_name in layer_by_chan_by_ratio:
    for channel in layer_by_chan_by_ratio[layer_name]:
        
        if layer_by_chan_by_ratio[layer_name][channel] > 0.9:
            very_sparse_neurons.append((layer_name, channel))

In [ ]:
very_sparse_neurons

In [ ]:
label_by_points = main_pw_defdict['mixed4d_1x1_pre_relu_conv']['89']
_ = plot_grid(label_by_points, 2,2)

In [ ]:
label_by_points = main_pw_defdict['mixed4d_pool_reduce_pre_relu_conv']['35']
_ = plot_grid(label_by_points, 4, 4)

In [ ]:
chan_by_ratio

In [ ]:
max_chan_by_score_in_4e = {}
for chan, w in enumerate(model.get_submodule("mixed4e_1x1_pre_relu_conv").weight):
    w = w.reshape(-1)
    max_neuron = torch.argmax(w.abs()).item()
    max_chan_by_score_in_4e[chan] = chan_by_ratio[max_neuron]
    

In [ ]:
len(chan_by_ratio), len(max_chan_by_score_in_4e)

In [ ]:
chan_by_ratio

In [ ]:
_, axes = plt.subplots(1, 2, sharey=True, figsize=(10,5))
axes[0].hist(np.random.choice(list(chan_by_ratio.values()), len(max_chan_by_score_in_4e)), bins=10)
axes[1].hist(max_chan_by_score_in_4e.values(), bins=10)
plt.show()

In [ ]:
# not much lol.   
plt.hist(chan_by_max_med.values(), bins=10)
plt.show()

In [ ]:
plt.hist(max_chan_by_score_in_4e.values(), bins=10)
plt.show()

# high activation inputs for cat

In [ ]:
# simple, we cluster with high acts of inputs of cat cluster, only these dims, we'll use the medians to find elbow
# its also easier to just find correlation plot btw, that will also be done.  
# the first is correlation plot

In [ ]:
# we would have a lot of pointwise mults and their correlation with our cat mult
# then we want to see how many are cats and not categorised as cats above a threshold
# first need to find a threshold, need to then run first
# i feel like ive done this exercise before

In [ ]:
ourdf = df[
    (df.layer_name == "mixed4e_1x1_pre_relu_conv") & 
    (df.cluster_label == 61) &
    (df.channel == 55)
]

w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().cpu().reshape(-1)


patches, pws = [], []

for ik  in tqdm(ourdf.input_image_key.unique()):
    timg = transform(Image.open(Path("flat-images") / f"{ik}.jpeg"))[None]
    acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])
    for tup in ourdf[ourdf.input_image_key == ik].itertuples():        
        ip = acts["mixed4e_1x1_pre_relu_conv"]["input"][0, :, tup.y_position, tup.x_position]
        patches.append(ip)
        pws.append(w*ip)
        

patches = torch.stack(patches)
pws = torch.stack(pws)

In [ ]:
pw_meds = pws.median(dim=0).values


In [ ]:
pw_meds[pw_meds > 0.01].shape

In [ ]:
thresh = 0.01

idxs = torch.argwhere(pw_meds > 0.01).reshape(-1)

In [ ]:
idxs.tolist()

In [ ]:
pw_meds = torch.sort(pw_meds).values
pw_meds[-80]

In [ ]:
plt.plot(torch.sort(pw_meds).values)
plt.show()

# Range ANDing


- cluster 61

In [ ]:
df.head()

In [ ]:
ys = main_act_dict["mixed4d_1x1_pre_relu_conv"]["73"]["16"]
xs = range(len(ys))
plt.scatter(xs, ys)
plt.show()

In [ ]:
df[(df.cluster_label == 61) & (df.layer_name == 'mixed4e_1x1_pre_relu_conv')].head()

In [ ]:
idf = df.set_index("input_image_key", drop=False)g

In [ ]:
w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().cpu().reshape(-1)

In [ ]:
image_keys = df[(df.cluster_label == 61) & (df.layer_name == 'mixed4e_1x1_pre_relu_conv')].input_image_key.unique()

In [ ]:
from sklearn.preprocessing import normalize
FLAT_IMAGE_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/flat-images"
)

normed_neuron_by_vals = defaultdict(list)
neuron_by_vals = defaultdict(list)
problems = []
for image_key in tqdm(image_keys):
    orig_key_df = idf.loc[[image_key]]
    key_df = orig_key_df[(orig_key_df.cluster_label == 61) & (orig_key_df.layer_name == "mixed4e_1x1_pre_relu_conv")]
    if len(key_df) == 0:
        continue
    timg = transform(Image.open(FLAT_IMAGE_DIR / f"{image_key}.jpeg"))[None]
    main_act = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv", "mixed4d_1x1_pre_relu_conv"])
    act = main_act["mixed4e_1x1_pre_relu_conv"]["input"]
    for tup in key_df.itertuples():
        normed_nvals = normalize((w * act[0, :, tup.y_position, tup.x_position]).reshape(1,-1), "l2")[0]
        nvals = w * act[0, :, tup.y_position, tup.x_position]

        coi = orig_key_df[
            # (orig_key_df.cluster_label == 29) & 
            (orig_key_df.layer_name == "mixed4d_pool_reduce_pre_relu_conv") & 
            # (orig_key_df.channel == 29) & 
            (orig_key_df.channel == 13) & 
            (orig_key_df.y_position == tup.y_position) &
            (orig_key_df.x_position == tup.x_position)
        ]
        if len(coi) > 0:
            print("found cluster, the val for this now is", nvals[493], "cluster label", coi.cluster_label.iloc[0])
        else:
            # whats the activation of the
            # print(main_act["mixed4d_1x1_pre_relu_conv"]["input"].shape, main_act["mixed4e_1x1_pre_relu_conv"]["input"].shape)
            problems.append(main_act["mixed4d_1x1_pre_relu_conv"]["input"][0, :, tup.y_position, tup.x_position])

        
        # print(nvals.shape)
        for i, v in enumerate(nvals):
            neuron_by_vals[i].append(v.item())
        for i, v in enumerate(normed_nvals):
            normed_neuron_by_vals[i].append(v.item())


In [ ]:
w_ = model.get_submodule("mixed4d_1x1_pre_relu_conv").weight[61].reshape(-1).detach()

In [ ]:
nps[0].shape

In [ ]:
nps = []
for p in problems[:5]:
    p = normalize(p.reshape(1,-1), "l2")
    p = p * normalize(w_.reshape(1,-1), "l2")
    
    nps.append(p[0])

In [ ]:
for i, p in enumerate([(p * w_).sum() for p in problems]):
    print(i,p)

In [ ]:
# well they do look the same
# there are many small differences everywhere...
# dictionary learning might be the real og i guess maybe?
S([(problems[p] * w_).reshape(1,-1).reshape(16,32) for p in [16, 0, 17, 12, 94]], 20, 5, viztype="local")

# pattern is definitely the same lol
# S([p.reshape(16,32) for p in nps], 20, 5)


In [ ]:
# 528 right? 

ys = neuron_by_vals[477]
xs = range(len(ys))
plt.scatter(xs, ys)

In [ ]:
# 528 right? 

ys = neuron_by_vals[493]
xs = range(len(ys))
plt.scatter(xs, ys)

In [ ]:
# 528 right? 

ys = neuron_by_vals[493]
xs = range(len(ys))
plt.scatter(xs, ys)

In [ ]:
with open("cat-neurons-description.json") as f:
    cat_neurons = json.load(f)

In [ ]:
{key: val for key, val in cat_neurons.items() if any(c.isupper() for c in val["meaning"])}

In [ ]:
nkeys = [val["key"] for key, val in cat_neurons.items() if not any(c.isupper() for c in val["meaning"])]

In [ ]:
useful_neurons = [_get_flattened_chan_in_cat_layer(layer_name, chan) for layer_name, chan, _ in nkeys]
useful_neurons[0]

In [ ]:
neuron = useful_neurons[10]
ys = neuron_by_vals[61]
xs = range(len(ys))
plt.scatter(xs, ys)
plt.title(neuron)
plt.show()

In [ ]:
neuron

In [ ]:
stds, norm_stds = [], []
xs = []
for i in range(528):
    # print(i, cat_neurons)
    
    if i in useful_neurons:
        xs.append(i)
        stds.append(np.std(neuron_by_vals[i]))
        norm_stds.append(np.std(normed_neuron_by_vals[i]))

In [ ]:
stds[493] = 0
norm_stds[493] = 0

In [ ]:
plt.plot(xs,stds)
plt.show()

In [ ]:
hehe = norm_stds.copy()
hehe[493] = 0
np.sum(hehe)

In [ ]:
norm_stds[493]

In [ ]:
norm_stds[9]

In [ ]:
xs[:15]

In [ ]:
plt.plot(xs[:15], norm_stds[:15])
plt.show()

In [ ]:
neuron = useful_neurons[25]


_, axes = plt.subplots(1, 2, figsize=(13,5))

ys1 = neuron_by_vals[neuron]
ys2 = normed_neuron_by_vals[neuron]
xs = range(len(ys1))

axes[0].scatter(xs, ys1)
axes[1].scatter(xs, ys2)

plt.show()

In [ ]:
# now, we would like to keep track of the ranges, we would have the positions to look for from df

key_df = idf.loc[[image_keys[0]]]
key_df[(key_df.cluster_label == 61) & (key_df.layer_name == "mixed4e_1x1_pre_relu_conv")]